# NB8 — the follow-up Stage-B arms

Trains the follow-up arms from each seed's verified M_A, paired with the main study's runs (same seed, data order, token budget, evaluation batches): tier 1 `protected_svd`, `protected_replay`; tier 2 `protected_replay_svd`, `sequential_svd`; tier 3 `protected_k256`. The first run also checks that this driver reproduces the main study's protected run. Dev scoring only at update 35 and the final update (update 0 is reused), no merges: about 1.9 GPU-hours per arm and seed, so one 12-hour session finishes tiers 1 and 2 for three seeds.

**Accelerator:** GPU T4 x2 · **Internet:** OFF · **Attach:** pcl-next code, model, NB2, all NB3, all NB4, NB7 (+ every earlier NB8 output)

Run with **Save Version → Save & Run All (Commit)**. Afterwards: if it says items are left, run it again with this output attached too. Then run NB9.

In [ ]:
SESSION_HOURS = 12   # your GPU session limit (see the notebook settings panel)
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})
# --- Follow-up add-on (pclx lives in notebooks_2/ of the same code dataset) ------------
NEXT = WORK / 'notebooks_2'
assert (NEXT / 'pclx' / 'cli.py').is_file(), 'attach the pcl-next code dataset (it contains notebooks_2/)'
sys.path.insert(0, str(NEXT))
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in (str(WORK), str(NEXT), os.environ.get('PYTHONPATH', '')) if p)
from pclx import plan

def pclx(*args):
    print('$ pclx', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pclx', *map(str, args)], check=True, cwd=WORK)

def pclx_on_gpus(commands, log_dir):
    """One pclx command per T4 (at most two), logs streamed live."""
    commands = [json.dumps([sys.executable, '-m', 'pclx', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

found_next = plan.discover_next([INPUT, WORKING], exclude=code_roots[0])
print({'stage_a_seeds': plan.stage_a_seeds(found_next), 'spectra': sorted(found_next['spectra']),
       'finished_items': sorted(k for k in found_next['items'] if plan.item_complete(found_next, *k))})

In [ ]:
MODEL = kaggle.one(found, 'models', 'pinned model Dataset (made from the NB1 output)')
M0_CONFIG = kaggle.m0_config(MODEL, WORKING / 'configs' / 'm0.yaml')
DATA = kaggle.one(found, 'data', 'NB2 output (data)')
PILOT = kaggle.one(found, 'pilot', 'NB2 output (pilot)')
SETTINGS = kaggle.study_settings(PILOT, WORKING / 'study.yaml')
settings = yaml.safe_load(SETTINGS.read_text())
assert settings['precision_validated'], 'The pilot did not validate FP16. Read pilot/pilot_report.md first.'
print(settings)

In [ ]:
TIERS = (1, 2, 3)        # drop 3 (or 2 and 3) to stop earlier; see notebooks_2/README.md for each tier's cost
FORCE_SVD_ARMS = False   # NB7's go/no-go rule decides whether the SVD-initialized arms run
decisions = sorted(INPUT.rglob('go_no_go.json'))
assert decisions, 'attach the NB7 output (spectrum/go_no_go.json)'
run_svd = FORCE_SVD_ARMS or read_json(decisions[-1])['run_svd_arms']
print('SVD-initialized arms:', 'running' if run_svd else 'skipped (NB7 rule)')
items = plan.pending_items(found_next, tiers=TIERS, run_svd_arms=run_svd)
print('pending:', items)
commands = [['queue', '--search', INPUT, '--exclude', code_roots[0], '--items', *queue, '--model-config', M0_CONFIG,
             '--settings', SETTINGS, '--data-root', DATA, '--session-seconds', remaining_seconds(),
             '--out', WORKING / 'next' / f'gpu{gpu}'] for gpu, queue in enumerate(plan.split(items))]
if commands:
    pclx_on_gpus(commands, WORKING / 'logs')
after = plan.discover_next([INPUT, WORKING], exclude=code_roots[0])
left = plan.pending_items(after, tiers=TIERS, run_svd_arms=run_svd)
for rep in after['replication']:
    print('replication check:', 'PASSED' if rep['passed'] else 'FAILED', {m: r['max_relative_difference'] for m, r in rep['modes'].items()})
print('Follow-up arms finished. Next: NB9.' if not left else f'Items left {left}: run NB8 again with this output attached.')